Undersampling + Residual BLock + Sliding Window CNN + LSTM (no Attention)

In [1]:
import pandas as pd
train_df = pd.read_csv(r"train.csv")
valid_df = pd.read_csv(r"val.csv")
test_df = pd.read_csv(r"test.csv")

train_df.sort_values(by='timestamp', inplace=True)
valid_df.sort_values(by='timestamp', inplace=True)
test_df.sort_values(by='timestamp', inplace=True)

cols_to_drop = ['timestamp', 'network_ips_dst', 'network_ips_src', 'network_ports_dst', 'network_ports_src']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [2]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # tạo mảng index cho step = 1 (dành riêng cho các class hiếm cần bảo toàn)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        # tạo mảng index theo step được truyền vào (cho các class còn lại)
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        
        # lặp qua từng class (Lấy từ step=1 để đảm bảo không sót class nào)
        classes = np.unique(window_labels_step1)
        rng = np.random.default_rng(seed=42)
        for c in classes:
            # ƯU TIÊN GIỮ NGUYÊN băm cửa sổ 1-1 cho Class 2 và 6
            if c in [100]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
                
            count = len(c_indices)
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                 
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [2, 6]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt tại index thực sự
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 1 
NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1) 
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, set Train Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giảm từ 95760 xuống 20000 cửa sổ.
Class 1: Giữ nguyên 1307 cửa sổ (step=1).
Class 2: Giữ nguyên 12639 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giữ nguyên 12894 cửa sổ (step=1).
Class 4: Giữ nguyên 5278 cửa sổ (step=1).
Class 5: Giữ nguyên 5643 cửa sổ (step=1).
Class 6: Giảm từ 23544 xuống 20000 cửa sổ.
Class 7: Giữ nguyên 1957 cửa sổ (step=1).
Tổng số cửa sổ sau khi lọc và Undersampling: 79718
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 20520 cửa sổ (step=1).
Class 1: Giữ nguyên 280 cửa sổ (step=1).
Class 2: Giữ nguyên 2708 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giữ nguyên 2763 cửa sổ (step=1).
Class 4: Giữ nguyên 1131 cửa sổ (step=1).
Class 5: Giữ nguyên 1209 cửa sổ (step=1).
Class 6: Giữ nguyên 5038 cửa sổ (

In [4]:
import torch
import torch.nn as nn

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# MODEL CNN-BiLSTM (ĐÃ LƯỢC BỎ ATTENTION)
class CNN_BiLSTM_NoAttention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_NoAttention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        # Đã loại bỏ self.attention
        
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) 
        
        # Đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # CHUẨN HOÁ LAYER NORM THEO TỪNG TIME-STEP
        out = self.layer_norm(out)
        
        # THAY THẾ ATTENTION: Sử dụng Global Average Pooling theo chiều sequence
        # Lấy trung bình tất cả các time-steps để nén về (Batch, hidden_size * 2)
        context_vector = torch.mean(out, dim=1)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

In [5]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
# tạo folder để lưu model
import os
model_dir = "model_final"
os.makedirs(model_dir, exist_ok=True)

model = CNN_BiLSTM_NoAttention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val macro f1 không cải thiện sau 2 epochs (lưu ý đổi mode thành 'max')
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    # nếu là epoch chẵn thì không đánh giá trên tập val để tiết kiệm thời gian, chỉ đánh giá trên tập val ở các epoch lẻ

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Đưa val_macro_f1 vào scheduler (trước đó là avg_val_loss)
    scheduler.step(val_macro_f1)
    
    # Lưu model dựa trên Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_v2.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Macro F1 sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, zero_division=0, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 0.8982, Train Macro F1: 0.7635 | Val Loss: 0.7556, Val Macro F1: 0.6844


Epoch [2/60] - Train Loss: 0.6128, Train Macro F1: 0.9446 | Val Loss: 0.6963, Val Macro F1: 0.7535


Epoch [3/60] - Train Loss: 0.5825, Train Macro F1: 0.9623 | Val Loss: 0.7750, Val Macro F1: 0.7012


Epoch [4/60] - Train Loss: 0.5658, Train Macro F1: 0.9683 | Val Loss: 0.8625, Val Macro F1: 0.6967


...

Epoch [33/60] - Train Loss: 0.5252, Train Macro F1: 0.9875 | Val Loss: 0.7969, Val Macro F1: 0.7085


Epoch [34/60] - Train Loss: 0.5250, Train Macro F1: 0.9870 | Val Loss: 0.7982, Val Macro F1: 0.7075


Epoch [35/60] - Train Loss: 0.5255, Train Macro F1: 0.9874 | Val Loss: 0.8035, Val Macro F1: 0.7055


Epoch [36/60] - Train Loss: 0.5254, Train Macro F1: 0.9883 | Val Loss: 0.8079, Val Macro F1: 0.7043

[Early Stopping] Model không cải thiện Macro F1 sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.9806    0.9953    0.9879     20520
           1     0.0000    0.0000    0.0000       281
           2     0.6024    0.4312    0.5026      2709
           3     0.5253    0.6200    0.5687      2763
           4     0.9878    0.9982    0.9930      1132
           5     0.9451    0.3132    0.4705      1210
           6     0.8961    1.0000    0.9452      5039
           7     0.5988    0.9810    0.7437       420

    accuracy                         0.8882     34074
   macro avg     0.6920    0.6674    0.6514     34074
weighted avg     0.8873    0.8882    0.8796     34074



In [6]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
# tạo folder để lưu model
import os
model_dir = "model_final"
os.makedirs(model_dir, exist_ok=True)

model = CNN_BiLSTM_NoAttention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val macro f1 không cải thiện sau 2 epochs (lưu ý đổi mode thành 'max')
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    # nếu là epoch chẵn thì không đánh giá trên tập val để tiết kiệm thời gian, chỉ đánh giá trên tập val ở các epoch lẻ

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Đưa val_macro_f1 vào scheduler (trước đó là avg_val_loss)
    scheduler.step(val_macro_f1)
    
    # Lưu model dựa trên Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_v2.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Macro F1 sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, zero_division=0, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 0.8716, Train Macro F1: 0.7811 | Val Loss: 0.6701, Val Macro F1: 0.7487


Epoch [2/60] - Train Loss: 0.6070, Train Macro F1: 0.9474 | Val Loss: 0.6807, Val Macro F1: 0.7608


Epoch [3/60] - Train Loss: 0.5755, Train Macro F1: 0.9649 | Val Loss: 0.6523, Val Macro F1: 0.7762


Epoch [4/60] - Train Loss: 0.5647, Train Macro F1: 0.9672 | Val Loss: 0.6999, Val Macro F1: 0.7653


...

Epoch [22/60] - Train Loss: 0.5228, Train Macro F1: 0.9877 | Val Loss: 0.8694, Val Macro F1: 0.7191


Epoch [23/60] - Train Loss: 0.5226, Train Macro F1: 0.9885 | Val Loss: 0.8722, Val Macro F1: 0.7265


Epoch [24/60] - Train Loss: 0.5222, Train Macro F1: 0.9885 | Val Loss: 0.8696, Val Macro F1: 0.7326


Epoch [25/60] - Train Loss: 0.5228, Train Macro F1: 0.9872 | Val Loss: 0.8610, Val Macro F1: 0.7340

[Early Stopping] Model không cải thiện Macro F1 sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.9983    0.9868    0.9925     20520
           1     0.0030    0.0036    0.0033       281
           2     0.4625    0.6032    0.5236      2709
           3     0.5266    0.4090    0.4604      2763
           4     0.9818    0.9991    0.9904      1132
           5     0.9689    0.3091    0.4687      1210
           6     0.8813    0.9994    0.9367      5039
           7     0.7780    0.9762    0.8659       420

    accuracy                         0.8794     34074
   macro avg     0.7001    0.6608    0.6552     34074
weighted avg     0.8877    0.8794    0.8755     34074



UnderSampling + Residual Block + CNN + LSTM + Attention (no Sliding Window)

In [7]:
import pandas as pd
train_df = pd.read_csv(r"train.csv")
valid_df = pd.read_csv(r"val.csv")
test_df = pd.read_csv(r"test.csv")

train_df.sort_values(by='timestamp', inplace=True)
valid_df.sort_values(by='timestamp', inplace=True)
test_df.sort_values(by='timestamp', inplace=True)

cols_to_drop = ['timestamp', 'network_ips_dst', 'network_ips_src', 'network_ports_dst', 'network_ports_src']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_dataset.X.shape[1]

# CƠ CHẾ ATTENTION THEO THỜI GIAN (Temporal Attention)
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# VŨ KHÍ MỚI: Squeeze-and-Excitation (SE Block) - Chống Drift Kênh Đặc Trưng
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # Global Average Pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        # Bộ tạo trọng số động
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # "Ngửi" toàn bộ cửa sổ để đánh giá độ tin cậy của từng kênh
        y = self.fc(y).view(b, c, 1)    # Tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # Bóp nghẹt kênh bị nhiễm Drift, Khuếch đại kênh chuẩn

# KHỐI RESIDUAL KẾT HỢP SE-BLOCK
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        # Nhúng cơ chế SE (Channel Attention)
        self.se = SEBlock1D(out_channels)
        
        # Đường tắt (Shortcut Connection)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        # CHỐT CHẶN SE-BLOCK: Lọc bỏ tín hiệu rác do Concept Drift gây ra trên các biến (Features) mỏng manh
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


# ĐẠI TU MODEL CNN-BiLSTM
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # Thay thế CNN thường bằng Khối Residual Tích hợp SE
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # Giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # LayerNorm (Chuẩn hóa biên độ)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) 
        
        # Đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        if x.size(-1) >=2:
            x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # CHUẨN HOÁ LAYER NORM THEO TỪNG TIME-STEP
        out = self.layer_norm(out)
        
        context_vector, attn_weights = self.attention(out)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out


In [9]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledDataset(Dataset):
    def __init__(self, df, max_samples_per_class=20000):
        # Trích xuất Features và Labels
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        
        print(f"Đang tính toán các mẫu hợp lệ và Undersampling...")
        
        # Mảng chứa tất cả index từ 0 đến len(df) - 1
        all_indices = np.arange(len(self.X))
        all_labels = self.y.numpy()
        
        self.valid_indices = []
        
        # Lặp qua từng class
        classes = np.unique(all_labels)
        rng = np.random.default_rng(seed=42)
        
        for c in classes:
            # Lấy tất cả index của các mẫu thuộc class c
            c_indices = all_indices[all_labels == c]
            count = len(c_indices)
            
            # Nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # Chọn ngẫu nhiên không hoàn lại
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} mẫu.")
            else:
                # Nếu ít hơn hoặc bằng ngưỡng thì giữ nguyên
                print(f"Class {c}: Giữ nguyên {count} mẫu.")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn khi train
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số mẫu sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # Lấy index thực sự đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # Lấy feature và label tại dòng (index) tương ứng
        sample_X = self.X[actual_idx].unsqueeze(0) 
        label_y = self.y[actual_idx]
        
        return sample_X, label_y

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 

BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, đưa mỗi sample riêng lẻ).")
train_dataset = UndersampledDataset(train_df, max_samples_per_class=MAX_SAMPLES)


print(f"Khởi tạo tập Val và Test (Không Undersampling, đưa mỗi sample riêng lẻ, giữ nguyên gốc)...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledDataset(valid_df, max_samples_per_class=10000000)
test_dataset = UndersampledDataset(test_df, max_samples_per_class=10000000)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, đưa mỗi sample riêng lẻ).
Đang tính toán các mẫu hợp lệ và Undersampling...
Class 0: Giảm từ 95760 xuống 20000 mẫu.
Class 1: Giữ nguyên 1307 mẫu.
Class 2: Giữ nguyên 12639 mẫu.
Class 3: Giữ nguyên 12894 mẫu.
Class 4: Giữ nguyên 5278 mẫu.
Class 5: Giữ nguyên 5643 mẫu.
Class 6: Giảm từ 23553 xuống 20000 mẫu.
Class 7: Giữ nguyên 1957 mẫu.
Tổng số mẫu sau khi lọc và Undersampling: 79718
Khởi tạo tập Val và Test (Không Undersampling, đưa mỗi sample riêng lẻ, giữ nguyên gốc)...
Đang tính toán các mẫu hợp lệ và Undersampling...
Class 0: Giữ nguyên 20520 mẫu.
Class 1: Giữ nguyên 280 mẫu.
Class 2: Giữ nguyên 2708 mẫu.
Class 3: Giữ nguyên 2763 mẫu.
Class 4: Giữ nguyên 1131 mẫu.
Class 5: Giữ nguyên 1209 mẫu.
Class 6: Giữ nguyên 5047 mẫu.
Class 7: Giữ nguyên 419 mẫu.
Tổng số mẫu sau khi lọc và Undersampling: 34077
Đang tính toán các mẫu hợp lệ và Undersampling...
Class 0: Giữ nguyên 20520 mẫu.
Class 1: Giữ nguyên 281 mẫu.
Class 2: Giữ nguyên 2709 mẫu.
Class 3:

In [11]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
# tạo folder để lưu model
import os
model_dir = "model_final"
os.makedirs(model_dir, exist_ok=True)

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=1).to(DEVICE)
# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val macro f1 không cải thiện sau 2 epochs (lưu ý đổi mode thành 'max')
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    # nếu là epoch chẵn thì không đánh giá trên tập val để tiết kiệm thời gian, chỉ đánh giá trên tập val ở các epoch lẻ

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Đưa val_macro_f1 vào scheduler (trước đó là avg_val_loss)
    scheduler.step(val_macro_f1)
    
    # Lưu model dựa trên Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_v2.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Macro F1 sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, zero_division=0, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 1.2237, Train Macro F1: 0.5745 | Val Loss: 0.9709, Val Macro F1: 0.6250


Epoch [2/60] - Train Loss: 0.9801, Train Macro F1: 0.7141 | Val Loss: 0.9196, Val Macro F1: 0.6523


Epoch [3/60] - Train Loss: 0.9426, Train Macro F1: 0.7341 | Val Loss: 0.9455, Val Macro F1: 0.6553


Epoch [4/60] - Train Loss: 0.9220, Train Macro F1: 0.7587 | Val Loss: 0.9046, Val Macro F1: 0.6590


...

Epoch [37/60] - Train Loss: 0.8234, Train Macro F1: 0.8458 | Val Loss: 0.9358, Val Macro F1: 0.6684


Epoch [38/60] - Train Loss: 0.8240, Train Macro F1: 0.8452 | Val Loss: 0.9351, Val Macro F1: 0.6719


Epoch [39/60] - Train Loss: 0.8232, Train Macro F1: 0.8466 | Val Loss: 0.9352, Val Macro F1: 0.6693


Epoch [40/60] - Train Loss: 0.8234, Train Macro F1: 0.8473 | Val Loss: 0.9389, Val Macro F1: 0.6656

[Early Stopping] Model không cải thiện Macro F1 sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.9329    0.9888    0.9600     20520
           1     0.1486    0.0391    0.0620       281
           2     0.4426    0.2403    0.3115      2709
           3     0.4907    0.5841    0.5334      2763
           4     0.7395    0.7898    0.7638      1132
           5     0.6200    0.3331    0.4333      1210
           6     0.9228    0.9614    0.9417      5048
           7     0.8635    0.7833    0.8215       420

    accuracy                         0.8522     34083
   macro avg     0.6451    0.5900    0.6034     34083
weighted avg     0.8317    0.8522    0.8368     34083



In [12]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
# tạo folder để lưu model
import os
model_dir = "model_final"
os.makedirs(model_dir, exist_ok=True)

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=1).to(DEVICE)
# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val macro f1 không cải thiện sau 2 epochs (lưu ý đổi mode thành 'max')
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    # nếu là epoch chẵn thì không đánh giá trên tập val để tiết kiệm thời gian, chỉ đánh giá trên tập val ở các epoch lẻ

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Đưa val_macro_f1 vào scheduler (trước đó là avg_val_loss)
    scheduler.step(val_macro_f1)
    
    # Lưu model dựa trên Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_v2.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Macro F1 sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, zero_division=0, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 1.2200, Train Macro F1: 0.5892 | Val Loss: 0.9973, Val Macro F1: 0.6431


Epoch [2/60] - Train Loss: 0.9879, Train Macro F1: 0.7093 | Val Loss: 0.8965, Val Macro F1: 0.6698


Epoch [3/60] - Train Loss: 0.9422, Train Macro F1: 0.7372 | Val Loss: 0.9310, Val Macro F1: 0.6758


Epoch [4/60] - Train Loss: 0.9251, Train Macro F1: 0.7445 | Val Loss: 0.9228, Val Macro F1: 0.6511


...

Epoch [30/60] - Train Loss: 0.8251, Train Macro F1: 0.8637 | Val Loss: 0.9603, Val Macro F1: 0.6448


Epoch [31/60] - Train Loss: 0.8246, Train Macro F1: 0.8646 | Val Loss: 0.9571, Val Macro F1: 0.6454


Epoch [32/60] - Train Loss: 0.8240, Train Macro F1: 0.8655 | Val Loss: 0.9568, Val Macro F1: 0.6458


Epoch [33/60] - Train Loss: 0.8242, Train Macro F1: 0.8644 | Val Loss: 0.9636, Val Macro F1: 0.6454

[Early Stopping] Model không cải thiện Macro F1 sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.9292    0.9966    0.9617     20520
           1     0.0000    0.0000    0.0000       281
           2     0.4413    0.2625    0.3292      2709
           3     0.5234    0.5516    0.5371      2763
           4     0.6609    0.7800    0.7156      1132
           5     0.5349    0.2595    0.3495      1210
           6     0.9313    0.9616    0.9462      5048
           7     0.8859    0.8500    0.8676       420

    accuracy                         0.8536     34083
   macro avg     0.6134    0.5827    0.5883     34083
weighted avg     0.8267    0.8536    0.8357     34083



Undersampling + CNN + LSTM + Attention + Sliding Window

In [29]:
import pandas as pd
train_df = pd.read_csv(r"train.csv")
valid_df = pd.read_csv(r"val.csv")
test_df = pd.read_csv(r"test.csv")

train_df.sort_values(by='timestamp', inplace=True)
valid_df.sort_values(by='timestamp', inplace=True)
test_df.sort_values(by='timestamp', inplace=True)

cols_to_drop = ['timestamp', 'network_ips_dst', 'network_ips_src', 'network_ports_dst', 'network_ports_src']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [30]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        self.X = torch.tensor(df.drop(columns=['label']).values, dtype=torch.float32)
        self.y = torch.tensor(df['label'].values, dtype=torch.long)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # tạo mảng index cho step = 1 (dành riêng cho các class hiếm cần bảo toàn)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        # tạo mảng index theo step được truyền vào (cho các class còn lại)
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.valid_indices = []
        
        # lặp qua từng class (Lấy từ step=1 để đảm bảo không sót class nào)
        classes = np.unique(window_labels_step1)
        rng = np.random.default_rng(seed=42)
        for c in classes:
            # ƯU TIÊN GIỮ NGUYÊN băm cửa sổ 1-1 cho Class 2 và 6
            if c in [100]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
                
            count = len(c_indices)
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                 
                c_indices = rng.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [2, 6]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)
            
        # Trộn đều danh sách index để các batch đa dạng hơn
        rng.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt tại index thực sự
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        return window_X, label_y

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 1 
NUM_FEATURES = train_df.shape[1] - 1
NUM_CLASSES = len(train_df['label'].unique())

# tối đa mỗi class sẽ có 20000 của sổ trượt
print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = UndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = UndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=1) 
test_dataset = UndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=1)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, set Train Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giảm từ 95760 xuống 20000 cửa sổ.
Class 1: Giữ nguyên 1307 cửa sổ (step=1).
Class 2: Giữ nguyên 12639 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giữ nguyên 12894 cửa sổ (step=1).
Class 4: Giữ nguyên 5278 cửa sổ (step=1).
Class 5: Giữ nguyên 5643 cửa sổ (step=1).
Class 6: Giảm từ 23544 xuống 20000 cửa sổ.
Class 7: Giữ nguyên 1957 cửa sổ (step=1).
Tổng số cửa sổ sau khi lọc và Undersampling: 79718
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Đang tính toán các cửa sổ hợp lệ (global step=1) và Undersampling...
Class 0: Giữ nguyên 20520 cửa sổ (step=1).
Class 1: Giữ nguyên 280 cửa sổ (step=1).
Class 2: Giữ nguyên 2708 cửa sổ (sử dụng step=1 để bảo toàn).
Class 3: Giữ nguyên 2763 cửa sổ (step=1).
Class 4: Giữ nguyên 1131 cửa sổ (step=1).
Class 5: Giữ nguyên 1209 cửa sổ (step=1).
Class 6: Giữ nguyên 5038 cửa sổ (

In [32]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        # lớp Linear để tính điểm số (score) cho từng time-step
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # đầu ra bi-lstm: (Batch, SeqLen, Hidden*2)
        
        # tính điểm chú tý cho mỗi timestap => (batch, seqlen, 1)
        scores = self.attention(lstm_outputs)
        
        # áp dung softmax để đưa sequence attention scores về dạng xác suất
        weights = torch.softmax(scores, dim=1)
        
        # nhân trọng số với output của lstm và tỉnh tổng (nhân chập)
        # (Batch, SeqLen, 1) * (Batch, SeqLen, Hidden*2) -> (Batch, Hidden*2)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        
        # trả lại context_vector và weights (để phục vụ cho việc trực quan hóa attention sau này)
        return context_vector, weights

# model CNN-BiLSTM với Cơ chế Attention
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # conv1d có padding=1 => giữ nguyên chiều dài chuỗi
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.dropout_cnn = nn.Dropout1d(0.2)

        # conv1d có padding=1 => giữ nguyên chiều dài chuỗi, tăng kênh lên 128
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        
        self.relu = nn.ReLU()
        
        # giảm 2 timesteps lại còn 1
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        # bi-lstm như cũ
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        

        # kích thước đầu vào của Attention = hidden_size * 2 (vì BiLSTM ghép 2 chiều ngược xuôi)
        self.attention = Attention(hidden_size * 2)
        
        self.dropout = nn.Dropout(0.5)
        
        # tầng phân loại cuối cùng
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # đầu vào (256,10, 314)
        
        # đảo trục cho CNN -> (256, 314, 10)
        x = x.permute(0, 2, 1) 
        
        # đi qua cnn block 1 => (256, 64, 10)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout_cnn(x)

        # đi qua cnn block 2 => (256, 128, 10)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        
        # giảm chiều dài chuỗi đi 2 lần => (256, 128, 5)
        x = self.pool(x)
        
        # đảo trục lại cho lstm (256, 5, 128)
        x = x.permute(0, 2, 1)
        
        # đi qua bi-lstm => (256, 5, 256)
        out, (hn, cn) = self.bilstm(x)
        
        # dùng attention để tính ra context vector: (256, 256)
        context_vector, attn_weights = self.attention(out)
        
        # đi qua lớp dense để ra dự đoán nhãn
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        # THAY ĐỔI DUY NHẤT Ở ĐÂY: in_channels = num_features (thay vì 1)
        # Conv1d sẽ trượt dọc theo chiều thời gian (Time_Steps) thay vì trượt dọc theo các features.
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        # tầng 2: giữ nguyên
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        # tầng 3: giữ nguyên
        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        # tầng BiLSTM: giữ nguyên
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        # tầng attention: giữ nguyên
        self.attention = Attention(128 * 2) 

        # các tầng dense: giữ nguyên
        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        # tầng phân loại cuối cùng: giữ nguyên
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Đầu vào x từ SlidingWindowDataset đang có shape: (Batch, Time_Steps, Features)
        
        # THAY ĐỔI Ở ĐÂY: Dùng permute thay vì unsqueeze
        # Chuyển vị để shape trở thành: (Batch, Features, Time_Steps)
        # Để Conv1d quét qua chiều Time_Steps với số kênh tương ứng với số Features
        x = x.permute(0, 2, 1) 

        # đi qua 3 tầng Conv1dCNN (Code đoạn này giữ nguyên 100%)
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        # permute để đưa qua BiLSTM: (Batch, Channels, Time_Steps_New) -> (Batch, Time_Steps_New, Channels)
        # Chiều Channels lúc này đang là 128 (khớp hoàn hảo với input_size=128 của BiLSTM)
        x = x.permute(0, 2, 1)

        # đi qua BiLSTM
        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        # đi qua attention để lấy context vector
        context_vector, _ = self.attention(out)

        # đi qua các lớp dense
        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        return out

In [34]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
# tạo folder để lưu model
import os
model_dir = "model_final"
os.makedirs(model_dir, exist_ok=True)

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val macro f1 không cải thiện sau 2 epochs (lưu ý đổi mode thành 'max')
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    # nếu là epoch chẵn thì không đánh giá trên tập val để tiết kiệm thời gian, chỉ đánh giá trên tập val ở các epoch lẻ

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Đưa val_macro_f1 vào scheduler (trước đó là avg_val_loss)
    scheduler.step(val_macro_f1)
    
    # Lưu model dựa trên Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_v2.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Macro F1 sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, zero_division=0, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 1.0447, Train Macro F1: 0.4749 | Val Loss: 0.7329, Val Macro F1: 0.7081


Epoch [2/60] - Train Loss: 0.6421, Train Macro F1: 0.8430 | Val Loss: 0.7121, Val Macro F1: 0.7200


Epoch [3/60] - Train Loss: 0.5848, Train Macro F1: 0.9399 | Val Loss: 0.8929, Val Macro F1: 0.6877


Epoch [4/60] - Train Loss: 0.5677, Train Macro F1: 0.9551 | Val Loss: 0.7543, Val Macro F1: 0.7332


...

Epoch [21/60] - Train Loss: 0.5086, Train Macro F1: 0.9830 | Val Loss: 0.8179, Val Macro F1: 0.7076


Epoch [22/60] - Train Loss: 0.5083, Train Macro F1: 0.9835 | Val Loss: 0.8117, Val Macro F1: 0.7065


Epoch [23/60] - Train Loss: 0.5075, Train Macro F1: 0.9839 | Val Loss: 0.8135, Val Macro F1: 0.7098


Epoch [24/60] - Train Loss: 0.5078, Train Macro F1: 0.9842 | Val Loss: 0.8117, Val Macro F1: 0.7081

[Early Stopping] Model không cải thiện Macro F1 sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.9744    0.9979    0.9860     20520
           1     0.0000    0.0000    0.0000       281
           2     0.6260    0.4393    0.5163      2709
           3     0.5766    0.8078    0.6729      2763
           4     0.9956    0.9956    0.9956      1132
           5     0.9738    0.3066    0.4664      1210
           6     0.9718    0.9996    0.9855      5039
           7     0.7930    0.9214    0.8524       420

    accuracy                         0.9045     34074
   macro avg     0.7389    0.6835    0.6844     34074
weighted avg     0.9045    0.9045    0.8953     34074



In [19]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
# tạo folder để lưu model
import os
model_dir = "model_final"
os.makedirs(model_dir, exist_ok=True)

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val macro f1 không cải thiện sau 2 epochs (lưu ý đổi mode thành 'max')
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    # nếu là epoch chẵn thì không đánh giá trên tập val để tiết kiệm thời gian, chỉ đánh giá trên tập val ở các epoch lẻ

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Đưa val_macro_f1 vào scheduler (trước đó là avg_val_loss)
    scheduler.step(val_macro_f1)
    
    # Lưu model dựa trên Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_v2.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Macro F1 sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, zero_division=0, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 1.0636, Train Macro F1: 0.4207 | Val Loss: 0.8413, Val Macro F1: 0.3676


Epoch [2/60] - Train Loss: 0.6779, Train Macro F1: 0.7937 | Val Loss: 0.7725, Val Macro F1: 0.7008


Epoch [3/60] - Train Loss: 0.5852, Train Macro F1: 0.9391 | Val Loss: 1.1596, Val Macro F1: 0.6428


Epoch [4/60] - Train Loss: 0.5674, Train Macro F1: 0.9539 | Val Loss: 1.1874, Val Macro F1: 0.6475


...

Epoch [29/60] - Train Loss: 0.5040, Train Macro F1: 0.9849 | Val Loss: 0.7957, Val Macro F1: 0.7151


Epoch [30/60] - Train Loss: 0.5041, Train Macro F1: 0.9832 | Val Loss: 0.7960, Val Macro F1: 0.7176


Epoch [31/60] - Train Loss: 0.5040, Train Macro F1: 0.9839 | Val Loss: 0.7962, Val Macro F1: 0.7157


Epoch [32/60] - Train Loss: 0.5040, Train Macro F1: 0.9854 | Val Loss: 0.7956, Val Macro F1: 0.7170

[Early Stopping] Model không cải thiện Macro F1 sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.9782    0.9976    0.9878     20520
           1     0.3908    0.7260    0.5081       281
           2     0.5765    0.4895    0.5294      2709
           3     0.6297    0.6609    0.6449      2763
           4     0.9956    0.9956    0.9956      1132
           5     0.9809    0.2967    0.4556      1210
           6     0.9497    1.0000    0.9742      5039
           7     0.6576    0.9738    0.7850       420

    accuracy                         0.9027     34074
   macro avg     0.7699    0.7675    0.7351     34074
weighted avg     0.9057    0.9027    0.8964     34074



Undersampling + CNN + LSTM + Attention (attention + no sliding window) (cấu trúc giống paper gốc)

In [20]:
import pandas as pd
train_df = pd.read_csv(r"train.csv")
valid_df = pd.read_csv(r"val.csv")
test_df = pd.read_csv(r"test.csv")

train_df.sort_values(by='timestamp', inplace=True)
valid_df.sort_values(by='timestamp', inplace=True)
test_df.sort_values(by='timestamp', inplace=True)

cols_to_drop = ['timestamp', 'network_ips_dst', 'network_ips_src', 'network_ports_dst', 'network_ports_src']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
NUM_FEATURES = train_df.shape[1] - 1 # trừ đi cột label
NUM_CLASSES = len(train_df['label'].unique()) # số class
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        # tính điểm chú ý
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        # Nhân trọng số và tính tổng
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# kiến trúc chính của paper
class Paper_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes):
        super(Paper_CNN_BiLSTM_Attention, self).__init__()
        
        # tầng 1: conv1d với 128 filters, kernel size = 5, padding = 2 để giũ nguyên chiều dài chuỗi
        # in_channels = 1 (Vì ta coi 314 features như 1 chuỗi dài 314 đơn vị) (đưa từng flow riêng lẻ vào)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(128)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.drop1 = nn.Dropout1d(0.3)

        # tầng 2: conv1d với 256 filters, kernel size 5, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3)
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(256)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.drop2 = nn.Dropout1d(0.3)

        # tầng 3: conv1d với 128 filters, kernel size 3, ReLU -> Batch Norm, MaxPool(2), Dropout(0.3)
        self.conv3 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(kernel_size=2)
        self.drop3 = nn.Dropout1d(0.3)

        # tầng BiLSTM: 128 units, Dropout(0.3)
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.drop_lstm = nn.Dropout(0.3)

        # tầng attention
        self.attention = Attention(128 * 2) # *2 vì là Bidirectional

        # các tầng dense
        self.fc1 = nn.Linear(128 * 2, 256)
        self.drop_fc1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.drop_fc2 = nn.Dropout(0.4)

        # tầng phân loại cuối cùng
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Đầu vào x từ StandardFlowDataset đang có shape: (Batch, Features)
        # đầu vào x có shape: (256, 314)
        
        # shape trở thành: (256, 1, 314)
        x = x.unsqueeze(1) 

        # đi qua 3 tầng Conv1dCNN
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        # permute để đưa qua BiLSTM
        x = x.permute(0, 2, 1)

        # đi qua BiLSTM
        out, _ = self.bilstm(x)
        out = self.drop_lstm(out)

        # đi qua attention để lấy context vector
        context_vector, _ = self.attention(out)

        # đi qua các lớp dense
        out = self.fc1(context_vector)
        out = F.relu(out)
        out = self.drop_fc1(out)

        out = self.fc2(out)
        out = F.relu(out)
        out = self.drop_fc2(out)

        out = self.fc3(out)
        
        # không dùng softmax
        return out

# khởi tạo mô hình và đẩy lên GPU
model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
print(model)

Paper_CNN_BiLSTM_Attention(
  (conv1): Conv1d(1, 128, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop1): Dropout1d(p=0.3, inplace=False)
  (conv2): Conv1d(128, 256, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop2): Dropout1d(p=0.3, inplace=False)
  (conv3): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn3): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop3): Dropout1d(p=0.3, inplace=False)
  (bilstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (drop_lstm): Dropout(p=0.3, inp

In [22]:
from torch.utils.data import Dataset
import numpy as np
class StandardFlowDataset(Dataset):
    def __init__(self, df, max_samples_per_class=None):
        self.X = []
        self.y = []
        
        # nhóm theo nhãn và áp dụng undersampling
        for class_name, group in df.groupby('label'):
            if max_samples_per_class and len(group) > max_samples_per_class:
                group = group.sample(n=max_samples_per_class, random_state=42)
            
            self.X.append(group.drop(columns=['label']).values)
            self.y.append(group['label'].values)
            
        self.X = np.vstack(self.X)
        self.y = np.concatenate(self.y)
        
        # xáo trộn dữ liệu sau khi đã áp dụng undersampling để đảm bảo tính ngẫu nhiên
        idx = np.random.permutation(len(self.X))
        self.X = torch.tensor(self.X[idx], dtype=torch.float32)
        self.y = torch.tensor(self.y[idx], dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# khởi tạo dataset với max_samples_per_class để giới hạn số lượng mẫu của mỗi class trong tập train, còn val và test thì giữ nguyên để đánh giá chính xác hơn
MAX_SAMPLES = 20000 
train_dataset = StandardFlowDataset(train_df, max_samples_per_class=MAX_SAMPLES)
val_dataset = StandardFlowDataset(valid_df)
test_dataset = StandardFlowDataset(test_df) 

BATCH_SIZE = 256
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [23]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
# tạo folder để lưu model
import os
model_dir = "model_final"
os.makedirs(model_dir, exist_ok=True)

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val macro f1 không cải thiện sau 2 epochs (lưu ý đổi mode thành 'max')
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    # nếu là epoch chẵn thì không đánh giá trên tập val để tiết kiệm thời gian, chỉ đánh giá trên tập val ở các epoch lẻ

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Đưa val_macro_f1 vào scheduler (trước đó là avg_val_loss)
    scheduler.step(val_macro_f1)
    
    # Lưu model dựa trên Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_v2.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Macro F1 sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, zero_division=0, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 1.4485, Train Macro F1: 0.3505 | Val Loss: 1.0004, Val Macro F1: 0.4454


Epoch [2/60] - Train Loss: 1.1952, Train Macro F1: 0.5509 | Val Loss: 0.9591, Val Macro F1: 0.5566


Epoch [3/60] - Train Loss: 1.1141, Train Macro F1: 0.6063 | Val Loss: 0.9750, Val Macro F1: 0.5491


Epoch [4/60] - Train Loss: 1.0419, Train Macro F1: 0.6586 | Val Loss: 0.8942, Val Macro F1: 0.6478


...

Epoch [37/60] - Train Loss: 0.8580, Train Macro F1: 0.7860 | Val Loss: 0.8989, Val Macro F1: 0.6972


Epoch [38/60] - Train Loss: 0.8590, Train Macro F1: 0.7825 | Val Loss: 0.8964, Val Macro F1: 0.6981


Epoch [39/60] - Train Loss: 0.8569, Train Macro F1: 0.7830 | Val Loss: 0.8969, Val Macro F1: 0.6977


Epoch [40/60] - Train Loss: 0.8579, Train Macro F1: 0.7875 | Val Loss: 0.8966, Val Macro F1: 0.6976

[Early Stopping] Model không cải thiện Macro F1 sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.9295    0.9934    0.9604     20520
           1     0.0000    0.0000    0.0000       281
           2     0.4462    0.2141    0.2893      2709
           3     0.5092    0.5722    0.5389      2763
           4     0.6494    0.7968    0.7156      1132
           5     0.5000    0.2810    0.3598      1210
           6     0.9294    0.9602    0.9446      5048
           7     0.7576    0.8262    0.7904       420

    accuracy                         0.8503     34083
   macro avg     0.5902    0.5805    0.5749     34083
weighted avg     0.8227    0.8503    0.8311     34083



In [24]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
# tạo folder để lưu model
import os
model_dir = "model_final"
os.makedirs(model_dir, exist_ok=True)

model = Paper_CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)
# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val macro f1 không cải thiện sau 2 epochs (lưu ý đổi mode thành 'max')
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    # nếu là epoch chẵn thì không đánh giá trên tập val để tiết kiệm thời gian, chỉ đánh giá trên tập val ở các epoch lẻ

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Đưa val_macro_f1 vào scheduler (trước đó là avg_val_loss)
    scheduler.step(val_macro_f1)
    
    # Lưu model dựa trên Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_v2.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Macro F1 sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_v2.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, zero_division=0, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 1.4458, Train Macro F1: 0.3848 | Val Loss: 0.9882, Val Macro F1: 0.5158


Epoch [2/60] - Train Loss: 1.1832, Train Macro F1: 0.5481 | Val Loss: 0.9223, Val Macro F1: 0.5005


Epoch [3/60] - Train Loss: 1.1039, Train Macro F1: 0.6075 | Val Loss: 0.9691, Val Macro F1: 0.6164


Epoch [4/60] - Train Loss: 1.0150, Train Macro F1: 0.6718 | Val Loss: 0.9108, Val Macro F1: 0.6350


...

Epoch [27/60] - Train Loss: 0.8641, Train Macro F1: 0.7863 | Val Loss: 0.9370, Val Macro F1: 0.6766


Epoch [28/60] - Train Loss: 0.8628, Train Macro F1: 0.7880 | Val Loss: 0.9302, Val Macro F1: 0.6788


Epoch [29/60] - Train Loss: 0.8642, Train Macro F1: 0.7870 | Val Loss: 0.9323, Val Macro F1: 0.6780


Epoch [30/60] - Train Loss: 0.8609, Train Macro F1: 0.7901 | Val Loss: 0.9297, Val Macro F1: 0.6780

[Early Stopping] Model không cải thiện Macro F1 sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.9245    0.9554    0.9397     20520
           1     0.0000    0.0000    0.0000       281
           2     0.6730    0.2743    0.3897      2709
           3     0.5771    0.6692    0.6197      2763
           4     0.6135    0.7783    0.6861      1132
           5     0.4344    0.2901    0.3479      1210
           6     0.8157    0.9602    0.8821      5048
           7     0.8750    0.8000    0.8358       420

    accuracy                         0.8395     34083
   macro avg     0.6142    0.5909    0.5876     34083
weighted avg     0.8243    0.8395    0.8231     34083

